In [ ]:
%pip install -U langchain-community langchain-text-splitters langchain-ollama chromadb beautifulsoup4

In [ ]:
%pip show langchain-community langchain-text-splitters langchain-ollama chromadb beautifulsoup4

In [18]:
import os
from getpass import getpass

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API key: ")
os.environ["LANGSMITH_PROJECT"] = "rag-qwen-local"

In [42]:
import langchain_community
import langchain_text_splitters
import langchain_ollama
import chromadb

print("Imports succeeded")

Imports succeeded


RAG Overview 

In [49]:
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("Local Ollama models configured")

Local Ollama models configured


In [50]:
### INDEXING ###

url = "https://sports.yahoo.com/articles/max-verstappen-reveals-problem-ruined-051513975.html?guccounter=1/"
loader = WebBaseLoader(web_paths=(url,))
docs = loader.load()

if not docs or not any(doc.page_content.strip() for doc in docs):
    raise ValueError(
        f"No page text was extracted from {url}. "
        "The site may require JavaScript or block automated requests."
    )

print(f"Loaded {len(docs)} document(s), {sum(len(doc.page_content) for doc in docs)} characters total.")


Loaded 1 document(s), 10499 characters total.


In [51]:
print("Source:", docs[0].metadata.get("source"))
print("Page content preview:\n", docs[0].page_content[:2000])

Source: https://sports.yahoo.com/articles/max-verstappen-reveals-problem-ruined-051513975.html?guccounter=1/
Page content preview:
 Max Verstappen Reveals the Problem That Ruined His RB22 in Brazil - Yahoo SportsSearch Query for SearchSearchHomeNewsFinanceSportsMoreMailRacingFormula 1NASCARIndycarMotoGP...FantasyNetworkFantasy PlusTRENDINGNFL trade deadline trackerBills beat Vikings on MNFWemby's on a tearNFL Power RankingsWeek 10 Waiver WireStory byMax Verstappen Reveals the Problem That Ruined His RB22 in BrazilStory byMax Verstappen Reveals the Problem That Ruined His RB22 in BrazilMariano Velloso·Roundtable SportsTue, November 10, 2026 at 5:15 AM UTC·3 min readAbout Our AdsRecommended StoriesHow Max Verstappen beat Red Bull's own simulation tools with P2 in BrazilTeam principal Laurent Mekies admits that Red Bull's simulation tools did not consider second place possible from 21st on the gridMotorsportVerstappen reveals what went wrong with RB22 despite impressive chargeMax Verstapp

In [52]:
# Split

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Requires a one-time `ollama pull nomic-embed-text` for local embeddings.
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

retriever = vectorstore.as_retriever()

In [53]:
print(type(retriever))
print(retriever)

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
tags=['Chroma', 'OllamaEmbeddings'] vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000245001FFD10> search_kwargs={}


In [55]:
### Retrieval And Generation ###

#Prompt

'''prompt = hub.pull("rlm/rag-prompt" , dangerously_pull_public_prompt=True ,)'''

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """Answer the question using the context below.
If the context doesn't contain the answer, say you don't know.
Keep the answer concise.

Question: {question}
Context: {context}
Answer:"""
)

#Post-processing

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#Chain

rag_chain = (
    {"context": retriever | format_docs, "question" : RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

#Question
rag_chain.invoke("What achievements of Max Verstappen are mentioned in the context? elaborate about him a bit")

'Based on the context, Max Verstappen is a four-time champion Dutch driver for Red Bull. His mentioned achievements include finishing second in the 2026 Sao Paulo Grand Prix after starting from 21st, securing his eighth consecutive podium, and placing 10th out of 11 outings. He is described as a "man for all seasons" who regularly beats simulations and delivers memorable performances in wet conditions at Interlagos. Team principal Laurent Mekies praised his remarkable recovery from the grid penalty to finish on the podium.'